# Stacking Ensemble Benchmark

Combining multiple models (LightGBM, XGBoost, CatBoost, RandomForest) using a Stacking Classifier.

## Import Libraries

Importing all the required libraries at the beginning in advance.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score

# Base Models
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

# Meta Model & Stacking
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier

## Load Preprocessed Data

Loading preprocessed data.

In [ ]:
# Reading the preprocessed dataset
processed_train = pd.read_parquet(
    "/kaggle/input/datasets/shivamgravity/pgs-s6e6-processed-data-v1/processed_train.parquet"
)
processed_test = pd.read_parquet(
    "/kaggle/input/datasets/shivamgravity/pgs-s6e6-processed-data-v1/processed_test.parquet"
)

## Configs

Explictly writing settings and parameters for further use.

In [ ]:
# Configs

# Target feature
TARGET = "class"
ID = "id"

# Test ids - used to create submission files after prediction
TEST_ID = processed_test[ID]

# Categorical columns
cat_cols = [
    "spectral_type",
    "galaxy_population"
]

# CV configs
N_SPLITS = 5
RANDOM_STATE = 42

## Data Preparation For Training & Testing

Splitting target feature from train dataset early, to manage the training further.

Removing the ID feature from train and test dataset both.

In [ ]:
# Preparing the datasets for training and testing purpose

# Removing the id and target feature
X = processed_train.drop([ID,TARGET], axis=1).copy()
y = processed_train[TARGET]

# Removing the id feature
X_test = processed_test.drop([ID], axis=1).copy()

## Encoding Categorical Feature

Encoding **TARGET** and **other categorical features**.

In [ ]:
# Encoding

# Encoding target feature
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y)

# Encoding categorical features beside target feature
feature_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

    feature_encoders[col] = le

## Stacking Configuration

Defining the Base Models and the Meta Model.

In [ ]:
# Define base models
# NOTE: Replace these with the best hyperparameters found in your individual experiments!
base_models = [
    ('lgbm', LGBMClassifier(random_state=RANDOM_STATE, n_estimators=500, verbosity=-1)),
    ('xgb', XGBClassifier(random_state=RANDOM_STATE, n_estimators=500, eval_metric='mlogloss')),
    ('cat', CatBoostClassifier(random_state=RANDOM_STATE, iterations=500, verbose=False)),
    ('rf', RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=300, n_jobs=-1))
]

# Define meta model
meta_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)

# Build the Stacking Classifier
stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=N_SPLITS,
    n_jobs=-1,
    passthrough=False
)

## 5-Fold Cross Validation

Evaluating the Stacking Classifier's overall ability using StratifiedKFold.

In [ ]:
fold_scores = []

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y[train_idx], y[valid_idx]
    
    model = StackingClassifier(
        estimators=base_models,
        final_estimator=meta_model,
        cv=3, # Internal CV for stacking
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    
    preds = model.predict(X_valid)
    score = balanced_accuracy_score(y_valid, preds)
    fold_scores.append(score)
    
    print(f"Fold {fold} Balanced Accuracy: {score:.4f}")

print(f"\nOverall CV Balanced Accuracy: {np.mean(fold_scores):.4f}")

## Final Model Training

Training the final stacking model on full data to predict on test dataset.

In [ ]:
final_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=N_SPLITS,
    n_jobs=-1
)
final_model.fit(X, y)

## Prediction On Test Data

The final trained model will be used to predict the test data.

In [ ]:
# Prediction on test data

# Predicting the values
pred = final_model.predict(X_test)

# Getting the associated labels with the numeric value predictions
pred_labels = target_encoder.inverse_transform(pred.astype(int))

## Competition Submission File

Saving the prediction as csv file to submit in the competition.

In [ ]:
# Saving the results

# Creating result dataframe
submission = pd.DataFrame({
    "id": TEST_ID,
    "class": pred_labels
})

# Saving the submission file
submission.to_csv("submission.csv", index=False)